In [1]:
# conda activate psix

import os
import sys
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

sys.path.append("code")

from modified_functions import *

In [2]:
# Load GTF

exclude = ""
gene_name = "gene_id"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


In [3]:
gtf.head()

,chrom,start,end,feature,strand,transcript,gene_type,gene,transcript_type,exon_id,exon_number,frame,tag,protein_id
0,chr1,11869,14409,gene,+,,lncRNA,ENSG00000290825,,,,0,overlaps_pseudogene,
1,chr1,11869,14409,transcript,+,ENST00000456328,lncRNA,ENSG00000290825,lncRNA,,,0,"basic,Ensembl_canonical",
2,chr1,11869,12227,exon,+,ENST00000456328,lncRNA,ENSG00000290825,lncRNA,ENSE00002234944,1,0,"basic,Ensembl_canonical",
3,chr1,12613,12721,exon,+,ENST00000456328,lncRNA,ENSG00000290825,lncRNA,ENSE00003582793,2,0,"basic,Ensembl_canonical",
4,chr1,13221,14409,exon,+,ENST00000456328,lncRNA,ENSG00000290825,lncRNA,ENSE00002312635,3,0,"basic,Ensembl_canonical",


## Get info. for all exons

In [4]:
# Load intron table (with junction coordinates per event)

intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file.tab.gz"
intron_table = pd.read_csv(intron_file, sep='\t', index_col=0)

intron_table_parsed = intron_table.copy()
coords = intron_table_parsed['intron'].str.split(':').str[1].str.split('-')
intron_table_parsed['intron_start'] = coords.str[0].astype(int)
intron_table_parsed['intron_end'] = coords.str[1].astype(int)

In [5]:
# Do this ONCE before the loop

gtf_exon = gtf[gtf.feature == "exon"]
gtf_indexed = gtf_exon.set_index(['chrom', 'start', 'end']).sort_index() # for exon lookup
exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup

gtf_cds = gtf[gtf.feature == "CDS"]
cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript lookup for coding sequences

In [6]:
# Compile list of all exons from cell type splicing analysis
exon_lookup = {}
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        signif_exons_df = signif_exons_df[~np.isnan(signif_exons_df['Oligo'])]  # keep as DataFrame, not just index
        
        for idx, row in signif_exons_df.iterrows():
            if idx not in exon_lookup:  # skip if already seen
                exon_lookup[idx] = {
                    "chr": row['chr'],
                    "exon_start": row['exon_start'],
                    "exon_end": row['exon_end']
                }

In [7]:
len(exon_lookup)

28837

In [8]:
def find_overlapping_cds(exon_start, exon_end, transcript_cds_rows):
    """Return the CDS row that overlaps this exon, or None."""
    for _, cds in transcript_cds_rows.iterrows():
        if cds['end'] >= exon_start and cds['start'] <= exon_end:
            return cds
    return None

In [9]:
def classify_overlap(exon_start, exon_end, cds_row):
    if cds_row['start'] == exon_start and cds_row['end'] == exon_end:
        return "fully_coding"
    elif cds_row['start'] >= exon_start and cds_row['end'] <= exon_end:
        return "partially_coding" # CDS subset of exon (should only happen when part of exon is UTR)
    else:
        return "unexpected"    # one side hangs over — rare

In [ ]:
exon_info = {}

# For each splicing event: save transcripts with compatible_transcripts splice junctions

for idx, val in tqdm(exon_lookup.items(), total=len(exon_lookup)):
    exon_start, exon_end, chrom = val['exon_start'], val['exon_end'], val['chr']
    target_exon = gtf_indexed.loc[(chrom, exon_start, exon_end)]
    strand = target_exon.strand.values[0]
    
    event_introns = intron_table_parsed[intron_table_parsed.event == idx]
    i1 = event_introns[event_introns.index.str.endswith("I1")]
    i2 = event_introns[event_introns.index.str.endswith("I2")]
    upstream_intron_start = i1.intron_start.values[0]
    downstream_intron_end = i2.intron_end.values[0]

    compatible_transcripts = {}

    for _, exon_row in target_exon.iterrows():
        target_transcript = exon_row['transcript']
        exon_number = int(exon_row['exon_number'])
        exon_id = exon_row['exon_id']
        all_exons_in_transcript = exons_by_transcript.get(target_transcript)

        # get flanking exons' end and start (direction depends on sense)
        upstream_num, downstream_num = (
            (exon_number + 1, exon_number - 1) if strand == "-" \
                    else (exon_number - 1, exon_number + 1)
        ) 
        te_indexed = all_exons_in_transcript.set_index(
            all_exons_in_transcript['exon_number'].astype(int)
        )
        
        if (upstream_num not in te_indexed.index) or (downstream_num not in te_indexed.index):
            continue        # exon is the first/last exon in this particular transcript (i.e. not a cassette exon)

        upstream_exon_end = te_indexed.loc[upstream_num, 'end'] + 1
        downstream_exon_start = te_indexed.loc[downstream_num, 'start'] - 1
        
        # check that exon junctions match target transcript:
        
        if (upstream_intron_start == upstream_exon_end) and (downstream_intron_end == downstream_exon_start):

            compatible_transcripts[target_transcript] = {
                "strand": strand,
                "transcript_type": all_exons_in_transcript.transcript_type.values[0], 
                "exon_number": exon_number,
                "exon_id": exon_id,
                "tag": all_exons_in_transcript.tag.values[0],
                "aa_start": '',
                "aa_end": '',
                "overlap_type": '',
                "coding_nt_length": '',
                "full_exon_nt_length": end - start + 1 
            }
            
            # if exon overlaps a coding region, get additional info
            
            transcript_cds = cds_by_transcript.get(target_transcript)
            
            if transcript_cds is None:
                continue    # transcript is noncoding
            
            overlapping_cds = find_overlapping_cds(exon_start, exon_end, transcript_cds)

            if overlapping_cds is None:
                continue   # exon is in UTR for this transcript
            
            # genomic coordinates of exon portion that OVERLAPS coding sequence
            exon_cds_start = overlapping_cds['start']
            exon_cds_end = overlapping_cds['end']
            
            cds_offset   = 0  
            gtf_frame = None
            for _, cds_row in transcript_cds.iterrows():
                if cds_row['start'] == exon_cds_start and cds_row['end'] == exon_cds_end:
                    gtf_frame = int(cds_row['frame'])
                    break   # reached our exon, stop
                cds_offset += cds_row['end'] - cds_row['start'] + 1
            
            # exon position relative to beginning of coding region
            # note: these coordinates describe the CODING PORTION of the exon
            rel_exon_start = cds_offset
            rel_exon_end = cds_offset + (exon_cds_end - exon_cds_start)

            # (first_frame - rel_exon_start): how many nts does the coding region UP TO target exon have (after accounting for incomplete upstream CDS)?
                                             # note: a first frame != 0 should only happen for transcripts with incomplete 5' annotations
            # (above) % 3: e.g. if above is divisble by 3, this should return frame 0
            first_frame = int(transcript_cds.iloc[0].frame) 
            this_cds_frame = (first_frame - rel_exon_start) % 3
            assert this_cds_frame == gtf_frame, (
                f"frame mismatch {target_transcript}: computed {this_cds_frame}, GTF {gtf_frame}"
            )
            
            aa_start = rel_exon_start // 3
            aa_end = rel_exon_end // 3
 
            # get position of amino acids encoded by the target exon
            compatible_transcripts[target_transcript].update({
                "aa_start": aa_start,
                "aa_end": aa_end,
                "overlap_type": classify_overlap(exon_start, exon_end, overlapping_cds),
                "coding_nt_length": cds_end - cds_start + 1
            })
         
    exon_info[idx] = compatible_transcripts
    # if len(compatible_transcripts) > 4:
    #     break

100%|██████████| 28837/28837 [06:51<00:00, 70.08it/s] 


In [19]:
rows = []
for idx, transcripts in exon_info.items():
    for transcript, info in transcripts.items():
        rows.append({
            'event': idx,
            'transcript': transcript,
            'transcript_type': info['transcript_type'],
            'exon_number': info['exon_number'],
            'tag': info['tag'],
            'aa_start': info['aa_start'],
            'aa_end': info['aa_end'],
            'overlap_type': info['overlap_type'],
            'coding_nt_length': info['coding_nt_length'],
            'full_exon_nt_length': info['full_exon_nt_length']
        })

exon_info_df = pd.DataFrame(rows).set_index('event')

In [20]:
exon_info_df.head()

,transcript,transcript_type,exon_number,tag,aa_start,aa_end,overlap_type,coding_nt_length,full_exon_nt_length
event,,,,,,,,,
ENSG00000107331_ProteinCoding_2,ENST00000341511,protein_coding,3,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",53,54,fully_coding,3,3
ENSG00000107331_ProteinCoding_2,ENST00000614293,protein_coding,3,"inferred_transcript_model,basic,GENCODE_Primar...",83,84,fully_coding,3,3
ENSG00000107331_ProteinCoding_2,ENST00000494046,retained_intron,3,mRNA_end_NF,,,,,3
ENSG00000107331_ProteinCoding_2,ENST00000476211,retained_intron,3,,,,,,3
ENSG00000277363_other_1,ENST00000621763,protein_coding_CDS_not_defined,17,NAGNAG_splice_site,,,,,123


In [21]:
exon_info_df.shape

(91360, 9)

## Log which transcripts were detected in RNA-seq data

In [ ]:
rsem_expr = pd.read_csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_RSEM_TPM.csv", index_col=0)
rsem_expr_subset = rsem_expr[rsem_expr.index.isin(exon_info_df['transcript'])]

### Calc isoform vs. ME corr

In [24]:
ctype_abund_df = pd.read_csv("data/ctype_abundance/GTEx_cortex_counts_TMMF_All_501_outliers_removed_top_Qval_mods_PC1_ctype_abundance_filtered_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)
ctype_abund_df.index = ctype_abund_df.index.str.replace(".", "-")

In [25]:
rsem_corr_results = {}
for ct in ctype_abund_df.columns:
    print(ct)
    rsem_corr_results[ct] = rsem_expr_subset.T.corrwith(ctype_abund_df[ct])

CGE Class
All GABAergic
Deep layer glutamatergic
All Neuronal
Oligo
Endo
Peri
OPC
Astro
Micro/PVM
VLMC
Upper layer glutamatergic


In [ ]:
rsem_corr_df = pd.DataFrame(rsem_corr_results)
rsem_corr_df.to_csv(f"data/corrs/GTEx_RSEM_TPM_ctype_abundance_corr.csv")

### Also save isoform mean expression

In [27]:
rsem_mean_expr = rsem_expr_subset.iloc[:, 1:].mean(axis=1)
is_detected = rsem_mean_expr.index.isin(exon_info_df.transcript)

In [28]:
rsem_info_df = pd.DataFrame({'RSEM_detected': is_detected, 'RSEM_mean_expr': rsem_mean_expr}, index=rsem_mean_expr.index)

## Now append exon info. to cell type exon analysis results

In [29]:
pd.set_option('display.max_columns', None)

In [30]:
column_order = ['Gene', 'is_specific', 'specific_direction', 
                'chr', 'strand', 'exon_start', 'exon_end', 'exon_len', 'transcript', 'transcript_type', 
                'exon_number', 'tag', 'coding_nt_length', 'full_exon_nt_length', 'overlap_type',  
                'aa_start', 'aa_end', 'RSEM_detected', 'RSEM_mean_expr', 'RSEM_expr_corr', 
                'r', 'fdr', 
                'CGE Class', 'All GABAergic', 'All Neuronal',
                'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
                'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
                ]

In [31]:
rsem_corr_df.columns = rsem_corr_df.columns.str.replace("/", "_").str.replace(" ", "_")

In [36]:
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ct = file.split("_exons.csv")[0]
        print(ct)
        
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        ctype_corr = rsem_corr_df[ct]
        ctype_corr.name = "RSEM_expr_corr"
        rsem_corr_info_df = rsem_info_df.merge(
            ctype_corr, left_index=True, right_index=True
        )
        exon_rsem_info_df = exon_info_df.merge(
            rsem_corr_info_df, left_on="transcript", right_index=True, how="left"
        )
        mask = exon_rsem_info_df.RSEM_detected == True 
        exon_rsem_info_df.loc[~mask, 'RSEM_detected'] = False
        
        signif_exons_info = signif_exons_df.merge(
            exon_rsem_info_df, 
            left_index=True, 
            right_index=True,
            how='left'
        )
        rest_columns = signif_exons_info.columns[signif_exons_info.columns.str.contains("diff")].tolist() 
        new_file = file.replace('_exons.csv', '_exons_annotated.csv')
        signif_exons_info[column_order + rest_columns].to_csv(f"data/ctype_exons/annotated/{new_file}")

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [41]:
pd.set_option('display.max_columns', None)